# 2022 Inventory Rerun Pipeline (Simplified)

**Purpose**: Reprocess 2022 EuropePMC dataset with latest production models using GPU acceleration  
**Created**: 2025-10-23  
**Updated**: 2025-10-28 (Simplified - removed checkpoints)  
**Environment**: Google Colab with GPU support  
**Status**: Production Pipeline with Mandatory Model Traceability  

---

## Overview

This notebook processes the 2022 EuropePMC dataset (21,677 papers) through a streamlined 5-step pipeline with:

- **Mandatory Model Traceability** - TRAINING_SESSION_ID required for full audit trail
- **Streamlined Processing** - Optimized for fixed 2022 dataset
- **GPU Acceleration** - 5-10x faster inference
- **No Checkpoints** - Simple linear pipeline (see PYTORCH_CHECKPOINT_FIX.md)
- **Google Drive Backup** - Final results archived to Drive
- **Clean Architecture** - Uses `src/rerun_utils.py` for all utility functions

## Pipeline Steps

1. **Input Validation** → Verify 2022 dataset and models
2. **Classification** → Identify bio-resource papers
3. **Named Entity Recognition** → Extract database names
4. **URL Extraction** → Find resource URLs
5. **Name Processing** → Generate final inventory

## Key Features

- ✅ **No Checkpoints**: Fresh run each time eliminates data contamination
- ✅ **Fast Execution**: ~10-15 minutes full run, ~5 minutes test mode
- ✅ **TEST_MODE Toggle**: Single variable for quick testing
- ✅ **Mandatory Traceability**: Full audit trail from training to inventory
- ✅ **Clean Cell Structure**: 11 cells following training notebook pattern

In [ ]:
# =============================================================================
# CELL 1: MOUNT GOOGLE DRIVE
# =============================================================================
# IMPORTANT: This MUST be the first cell for archive access

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("📦 Archive paths are now accessible")
print("🔗 Ready for streamlined processing")

In [ ]:
# =============================================================================
# CELL 2: RERUN PIPELINE CONFIGURATION
# =============================================================================

import os
import random
import string
from datetime import datetime

# =============================================================================
# MODE SELECTION
# =============================================================================
TEST_MODE = False  # Set to True for quick testing (~5 minutes with 1,000 papers)
                   # Set to False for full production rerun (~10-15 minutes with 21,677 papers)

# =============================================================================
# USER-EDITABLE CONFIGURATION
# =============================================================================

# REQUIRED: Model Traceability - Link to specific training session
TRAINING_SESSION_ID = ""  # e.g., "2025-10-23-abc123" from training notebook
# This MUST match the UNIQUE_ID from your training notebook run

# Input Data Configuration
INPUT_DATA = "data/epmc_query_results_2022.csv"  # Fixed: 2022 dataset

# Processing Configuration
MAX_URLS = 3  # Maximum URLs to extract per paper

# Mode-specific configuration
if TEST_MODE:
    TEST_SUBSET_SIZE = 1000
    print("🧪 TEST MODE ENABLED")
    print("   ⏱️  Expected runtime: ~5 minutes")
    print("   📊 Dataset: 1,000 papers (subset)")
else:
    TEST_SUBSET_SIZE = None
    print("🚀 PRODUCTION MODE ENABLED")
    print("   ⏱️  Expected runtime: ~10-15 minutes")
    print("   📊 Dataset: 21,677 papers (full 2022 dataset)")

# =============================================================================
# AUTO-GENERATED CONFIGURATION (DO NOT EDIT BELOW THIS LINE)
# =============================================================================

# Session Management
mode_suffix = "_test" if TEST_MODE else ""
RERUN_SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}{mode_suffix}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_DATE = datetime.now().strftime('%Y-%m-%d')

# Path Configuration
INVENTORY_DIRECTORY = "/content/drive/MyDrive/inventory_2022"
DATA_DIRECTORY = f"{INVENTORY_DIRECTORY}/data"

# Model Archive Paths
TRAINING_ARCHIVE_BASE = f"{INVENTORY_DIRECTORY}/training_archives/{TRAINING_SESSION_ID}_full_training"
ARCHIVE_CLASSIF_MODEL = f"{TRAINING_ARCHIVE_BASE}/classification_model.pt"
ARCHIVE_NER_MODEL = f"{TRAINING_ARCHIVE_BASE}/ner_model.pt"

# Working Model Locations (where scripts expect them)
TARGET_CLASSIF_MODEL = "out/classif_train_out/article_classifier.pt"
TARGET_NER_MODEL = "out/ner_train_out/named_entity_recognition.pt"

# Output Configuration
OUTPUT_BASE_DIR = "inventory_classification_results"
OUTPUT_RUN_DIR = f"{OUTPUT_BASE_DIR}/{RUN_DATE}_2022_rerun"
CLASSIF_DIR = f"{OUTPUT_RUN_DIR}/classification"
NER_DIR = f"{OUTPUT_RUN_DIR}/ner"
URL_DIR = f"{OUTPUT_RUN_DIR}/url_extraction"
NAMES_DIR = f"{OUTPUT_RUN_DIR}/processed_names"
FINAL_DIR = f"{OUTPUT_RUN_DIR}/final_results"
LOG_DIR = f"{OUTPUT_RUN_DIR}/logs"

# Results Archive
RESULTS_ARCHIVE_BASE = f"{INVENTORY_DIRECTORY}/rerun_results/{RERUN_SESSION_ID}_2022_rerun"

# Results file paths
CLASSIF_RESULTS = f"{CLASSIF_DIR}/predictions.csv"
CLASSIF_POSITIVES = f"{CLASSIF_DIR}/predicted_positives.csv"
NER_RESULTS = f"{NER_DIR}/predictions.csv"
URL_RESULTS = f"{URL_DIR}/predictions.csv"
NAMES_RESULTS = f"{NAMES_DIR}/predictions.csv"
FINAL_RESULTS = f"{FINAL_DIR}/biodata_inventory_2022_rerun.csv"

# Environment
os.environ['PYTHONPATH'] = 'src'

# Configuration dictionary for utilities
config = {
    'rerun_session_id': RERUN_SESSION_ID,
    'training_session_id': TRAINING_SESSION_ID,
    'timestamp': TIMESTAMP,
    'test_mode': TEST_MODE,
    'input_data': INPUT_DATA,
    'max_urls': MAX_URLS,
    'inventory_directory': INVENTORY_DIRECTORY,
    'data_directory': DATA_DIRECTORY,
    'training_archive_base': TRAINING_ARCHIVE_BASE,
    'results_archive_base': RESULTS_ARCHIVE_BASE,
    'created': datetime.now().isoformat()
}

if TEST_MODE:
    config['test_subset_size'] = TEST_SUBSET_SIZE

print("\n" + "=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)

In [ ]:
# =============================================================================
# CELL 3: VALIDATION & DISPLAY
# =============================================================================

import sys
from pathlib import Path

# Setup Python path
if f'{INVENTORY_DIRECTORY}/' not in sys.path:
    sys.path.append(f'{INVENTORY_DIRECTORY}/')

# Import utilities
from src.rerun_utils import validate_rerun_config, display_rerun_config

# Validate configuration
validate_rerun_config(config)

# Display configuration
display_rerun_config(config)

print("\n" + "=" * 60)
print("CONFIGURATION VALIDATED")
print("=" * 60)

In [ ]:
# =============================================================================
# CELL 4: ENVIRONMENT SETUP
# =============================================================================

print("🔧 Installing packages...")
!pip install transformers datasets evaluate seqeval nltk

print("\n✅ Dependencies installed")

# Download NLTK data
import nltk
import ssl

print("Downloading NLTK data...")
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')
print("✅ NLTK data downloaded")

# Import all utilities
from src.rerun_utils import *

# Import libraries
import pandas as pd
import numpy as np
import torch
import transformers
import shutil
import time
import json

# Verify environment
print(f"\n🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🤗 Transformers: {transformers.__version__}")
print(f"🎯 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

print("\n✅ Environment setup complete")

In [ ]:
# =============================================================================
# CELL 5: GPU CHECK
# =============================================================================

print("🔍 GPU Environment Check:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print("✅ GPU acceleration available")
else:
    print("⚠️ No GPU detected - inference will be slower")
    print("Consider enabling GPU runtime: Runtime > Change runtime type > GPU")

In [ ]:
# =============================================================================
# CELL 6: PROGRESS TRACKING SETUP
# =============================================================================

print("📊 Initializing progress tracking...")

# Initialize progress tracker (no checkpoint persistence)
progress = {
    "input_validation": "⏳",
    "classification": "⏳",
    "ner_processing": "⏳",
    "url_extraction": "⏳",
    "name_processing": "⏳",
    "final_results": "⏳"
}

show_rerun_progress(progress, RERUN_SESSION_ID)
print("✅ Ready to begin processing!")

In [ ]:
# =============================================================================
# CELL 7: MODEL LOADING WITH TRACEABILITY
# =============================================================================

print("🤖 Model Loading & Traceability System")
print("=" * 50)

# Create target directories
Path("out/classif_train_out").mkdir(parents=True, exist_ok=True)
Path("out/ner_train_out").mkdir(parents=True, exist_ok=True)

# Load models with traceability and verification
model_source, training_session_used, verification_report = load_models_with_traceability(
    TRAINING_SESSION_ID,
    INVENTORY_DIRECTORY,
    TARGET_CLASSIF_MODEL,
    TARGET_NER_MODEL
)

# Update config with traceability
config['model_traceability'] = {
    'model_source': model_source,
    'training_session_used': training_session_used,
    'models_loaded_at': datetime.now().isoformat(),
    'verification_report': verification_report
}

print(f"\n🎯 Model Traceability Established:")
print(f"   Training Session → {training_session_used}")
print(f"   Rerun Session → {RERUN_SESSION_ID}")
print(f"   Model Source → {model_source}")
if verification_report.get('all_verified'):
    print(f"   Checksum Verification → ✅ PASSED")
else:
    print(f"   Checksum Verification → ⚠️ Skipped (manifest not found)")
print("\n✅ Models ready for inference!")

In [ ]:
# =============================================================================
# CELL 8: INPUT VALIDATION
# =============================================================================

print("📚 Input Data Validation")
print("=" * 50)

# Create output directories
output_dirs = [OUTPUT_RUN_DIR, CLASSIF_DIR, NER_DIR, URL_DIR, NAMES_DIR, FINAL_DIR, LOG_DIR]
for dir_path in output_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

# Validate input data
valid, total_papers, columns = validate_input_data(
    f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}",
    ['id', 'title', 'abstract']
)

if not valid:
    raise ValueError(f"Input data validation failed for {INPUT_DATA}")

print(f"✅ Input data validated: {total_papers:,} papers")
print(f"📋 Columns: {columns}")

# Handle test mode
if TEST_MODE:
    print(f"\n🧪 Test Mode: Using subset of {TEST_SUBSET_SIZE} papers")
    input_df = pd.read_csv(f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}").head(TEST_SUBSET_SIZE)
    test_input_path = f"{OUTPUT_RUN_DIR}/test_input.csv"
    input_df.to_csv(test_input_path, index=False)
    effective_input = test_input_path
    papers_to_process = len(input_df)
    print(f"📁 Test subset saved: {test_input_path}")
else:
    effective_input = f"{DATA_DIRECTORY}/{INPUT_DATA.split('/')[-1]}"
    papers_to_process = total_papers
    print(f"\n🔍 Full Mode: Processing all {papers_to_process:,} papers")

progress["input_validation"] = "✅"
show_rerun_progress(progress, RERUN_SESSION_ID)
print("\n✅ Input validation complete")

In [ ]:
# =============================================================================
# CELL 9: CLASSIFICATION PIPELINE
# =============================================================================

progress['classification'] = '🔄'
show_rerun_progress(progress, RERUN_SESSION_ID)

print("📋 Step 1/5: Classification Pipeline")
print("=" * 40)

# Check if already completed (for notebook re-runs)
if Path(CLASSIF_RESULTS).exists() and Path(CLASSIF_POSITIVES).exists():
    df_all = pd.read_csv(CLASSIF_RESULTS)
    positives = pd.read_csv(CLASSIF_POSITIVES)
    print(f"✅ Classification already completed: {len(df_all):,} papers")
    print(f"📊 Bio-resource papers: {len(positives):,} ({len(positives)/len(df_all)*100:.1f}%)")
    progress['classification'] = '✅'
else:
    print("🚀 Running classification...")
    print(f"📅 Input: {effective_input}")
    print(f"🤖 Model: {TARGET_CLASSIF_MODEL}")

    start_time = time.time()

    # Run classification
    success, duration = run_prediction_script(
        'class_predict',
        INVENTORY_DIRECTORY,
        {
            '-i': effective_input,
            '-o': CLASSIF_DIR,
            '-c': TARGET_CLASSIF_MODEL
        }
    )

    if success:
        # Filter positives
        df_all = pd.read_csv(CLASSIF_RESULTS)
        positives = df_all[df_all['predicted_label'] == 'bio-resource']
        positives.to_csv(CLASSIF_POSITIVES, index=False)

        duration_mins = int(duration // 60)
        duration_secs = int(duration % 60)
        print(f"✅ Classification completed in {duration_mins}m {duration_secs}s")
        print(f"📊 Total: {len(df_all):,}, Bio-resource: {len(positives):,} ({len(positives)/len(df_all)*100:.1f}%)")

        progress['classification'] = '✅'
    else:
        raise RuntimeError("Classification processing failed")

show_rerun_progress(progress, RERUN_SESSION_ID)
print("🎯 Classification step complete!")

In [ ]:
# =============================================================================
# CELL 10: NER PIPELINE
# =============================================================================

progress['ner_processing'] = '🔄'
show_rerun_progress(progress, RERUN_SESSION_ID)

print("🏷️ Step 2/5: NER Pipeline")
print("=" * 40)

# Check if already completed (for notebook re-runs)
if Path(NER_RESULTS).exists():
    df = pd.read_csv(NER_RESULTS)
    print(f"✅ NER already completed: {len(df):,} papers")
    progress['ner_processing'] = '✅'
else:
    print("🚀 Running NER...")
    print(f"📅 Input: {CLASSIF_POSITIVES}")
    print(f"🤖 Model: {TARGET_NER_MODEL}")

    # Verify input exists
    if not Path(CLASSIF_POSITIVES).exists():
        raise FileNotFoundError(f"Bio-resource papers file not found: {CLASSIF_POSITIVES}")

    input_df = pd.read_csv(CLASSIF_POSITIVES)
    print(f"📊 Bio-resource papers to process: {len(input_df):,}")

    # Run NER
    success, duration = run_prediction_script(
        'ner_predict',
        INVENTORY_DIRECTORY,
        {
            '-i': CLASSIF_POSITIVES,
            '-o': NER_DIR,
            '-c': TARGET_NER_MODEL
        }
    )

    if success:
        df = pd.read_csv(NER_RESULTS)
        duration_mins = int(duration // 60)
        duration_secs = int(duration % 60)
        print(f"✅ NER completed in {duration_mins}m {duration_secs}s")
        print(f"📊 Papers with NER results: {len(df):,}")

        progress['ner_processing'] = '✅'
    else:
        raise RuntimeError("NER processing failed")

show_rerun_progress(progress, RERUN_SESSION_ID)
print("🎯 NER step complete!")

In [ ]:
# =============================================================================
# CELL 11: POST-PROCESSING & FINAL RESULTS
# =============================================================================

print("🔍 Step 3-5/5: Post-Processing Pipeline")
print("=" * 40)

# URL Extraction
print("\n📋 Step 3/5: URL Extraction")
if Path(URL_RESULTS).exists():
    df = pd.read_csv(URL_RESULTS)
    print(f"✅ URL extraction already completed: {len(df):,} entries")
else:
    print("🚀 Starting URL extraction...")
    success, duration = run_prediction_script(
        'url_extractor',
        INVENTORY_DIRECTORY,
        {
            NER_RESULTS: None,  # positional arg
            '-o': URL_DIR,
            '-x': str(MAX_URLS)
        }
    )

    if success:
        df = pd.read_csv(URL_RESULTS)
        print(f"✅ URL extraction completed in {duration:.1f}s")
        print(f"📊 Papers with URLs: {len(df):,}")
    else:
        raise RuntimeError("URL extraction failed")

progress["url_extraction"] = "✅"

# Name Processing
print("\n📋 Step 4/5: Name Processing")
if Path(NAMES_RESULTS).exists():
    df = pd.read_csv(NAMES_RESULTS)
    print(f"✅ Name processing already completed: {len(df):,} entries")
else:
    print("🚀 Starting name processing...")
    success, duration = run_prediction_script(
        'process_names',
        INVENTORY_DIRECTORY,
        {
            URL_RESULTS: None,  # positional arg
            '-o': NAMES_DIR
        }
    )

    if success:
        df = pd.read_csv(NAMES_RESULTS)
        print(f"✅ Name processing completed in {duration:.1f}s")
        print(f"📊 Processed entries: {len(df):,}")
    else:
        raise RuntimeError("Name processing failed")

progress["name_processing"] = "✅"

# Create final inventory
print("\n📋 Step 5/5: Creating final inventory...")
if Path(NAMES_RESULTS).exists():
    shutil.copy2(NAMES_RESULTS, FINAL_RESULTS)
    final_df = pd.read_csv(FINAL_RESULTS)
    final_count = len(final_df)
    print(f"✅ Final inventory created: {final_count:,} biodata resources")
    print(f"📁 Location: {FINAL_RESULTS}")
else:
    raise FileNotFoundError(f"Name processing results not found: {NAMES_RESULTS}")

progress["final_results"] = "✅"

# Create comprehensive archive
print(f"\n💾 Creating comprehensive archive...")
print(f"📦 Archive location: {RESULTS_ARCHIVE_BASE}")

archived_count = create_rerun_archive(
    RESULTS_ARCHIVE_BASE,
    RERUN_SESSION_ID,
    config,
    {
        'classification': CLASSIF_DIR,
        'ner': NER_DIR,
        'url_extraction': URL_DIR,
        'names': NAMES_DIR,
        'final': FINAL_DIR
    }
)

print(f"\n✅ Archive created with {archived_count} items")

# Final summary
completion_time = datetime.now()
processing_time = completion_time - datetime.fromisoformat(config['created'])
processing_mins = int(processing_time.total_seconds() / 60)
processing_hours = processing_mins // 60
processing_mins_remainder = processing_mins % 60

print("\n" + "🎉" * 20)
print("🎉 2022 INVENTORY RERUN COMPLETE")
print("🎉" * 20)

show_rerun_progress(progress, RERUN_SESSION_ID)

print(f"\n📊 Final Status:")
print(f"   Rerun Session ID: {RERUN_SESSION_ID}")
print(f"   Training Session Used: {training_session_used}")
print(f"   Completion Time: {completion_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Processing Time: {processing_hours}h {processing_mins_remainder}m")
print(f"   Papers Processed: {papers_to_process:,}")
print(f"   Final Inventory: {final_count:,} resources")

print(f"\n📁 Results Archive:")
print(f"   📁 {RESULTS_ARCHIVE_BASE}")
print(f"   📄 Main Output: final_inventory.csv")
print(f"   📄 Configuration: config_with_traceability.json")
print(f"   📄 Documentation: README.md")

print(f"\n🎯 Model Traceability:")
print(f"   Training Session → {training_session_used}")
print(f"   Rerun Session → {RERUN_SESSION_ID}")
print(f"   Models Source → {model_source}")
print(f"   Archive Location → {RESULTS_ARCHIVE_BASE}")

print(f"\n✨ Rerun completed successfully!")
print(f"🆔 Session ID: {RERUN_SESSION_ID} (save this for future reference)")
print(f"🔗 Training Session: {training_session_used} (linked models)")

print(f"\n🏁 Completed at {completion_time.strftime('%H:%M:%S')}")
print("📋 Full traceability chain established from training to final inventory!")
print("🛡️ Data integrity guaranteed - no checkpoint contamination")